In [0]:
from pyspark.sql import functions as F

# 1. Leer la tabla limpia y Gobernada desde la capa Silver
df_silver = spark.read.table("workspace.sch_oews_medallion.tb_slv_oews_clean")

# 2. Tabla Gold 1: Inteligencia Salarial para Roles de Datos y Tecnología (Código SOC empieza con '15-')
df_gold_tech_roles = (
    df_silver
    .filter(F.col("OCC_CODE").startswith("15-"))
    .select(
        F.col("AREA_TITLE").alias("area_metropolitana"),
        F.col("PRIM_STATE").alias("estado"),
        F.col("OCC_CODE").alias("codigo_ocupacion"),
        F.col("OCC_TITLE").alias("titulo_rol"),
        F.col("tot_emp_num").alias("total_empleados"),
        F.col("a_mean_num").alias("salario_anual_promedio"),
        F.col("h_mean_num").alias("salario_hora_promedio")
    )
    .orderBy(F.desc("salario_anual_promedio"))
)

# 3. Tabla Gold 2: Resumen Ejecutivo por Estado (Todos los sectores)
df_gold_state_summary = (
    df_silver
    .groupBy(F.col("PRIM_STATE").alias("estado"))
    .agg(
        F.sum("tot_emp_num").alias("empleo_total_estimado"),
        F.round(F.avg("a_mean_num"), 2).alias("salario_anual_medio_estado"),
        F.round(F.max("a_mean_num"), 2).alias("salario_maximo_registrado"),
        F.countDistinct("OCC_CODE").alias("cantidad_profesiones_medidas")
    )
    .orderBy(F.desc("salario_anual_medio_estado"))
)
# 4. Guardar las tablas Gold en Unity Catalog
df_gold_tech_roles.write.format("delta").mode("overwrite").saveAsTable("workspace.sch_oews_medallion.tb_gld_tech_salaries")
df_gold_state_summary.write.format("delta").mode("overwrite").saveAsTable("workspace.sch_oews_medallion.tb_gld_state_summary")

print("¡Capa Gold generada con éxito! Top 5 de roles tecnológicos mejor pagados:")
display(df_gold_tech_roles.limit(5))


¡Capa Gold generada con éxito! Top 5 de roles tecnológicos mejor pagados:


area_metropolitana,estado,codigo_ocupacion,titulo_rol,total_empleados,salario_anual_promedio,salario_hora_promedio
"Portland-Vancouver-Hillsboro, OR-WA",OR,15-1221,Computer and Information Research Scientists,660,223160.0,107.29
"San Jose-Sunnyvale-Santa Clara, CA",CA,15-1252,Software Developers,87350,221710.0,106.59
"San Jose-Sunnyvale-Santa Clara, CA",CA,15-1221,Computer and Information Research Scientists,2210,216560.0,104.12
"San Jose-Sunnyvale-Santa Clara, CA",CA,15-2051,Data Scientists,6060,212760.0,102.29
"Seattle-Tacoma-Bellevue, WA",WA,15-1221,Computer and Information Research Scientists,1620,203260.0,97.72
